In [ ]:
import os
import json
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm import tqdm
from pathlib import Path
import sympy

In [ ]:
# ==========================================
# 1. HELPER FUNCTIONS FOR EXTRACTION
# ==========================================
def extract_json_fields(text):
    """
    Safely extracts 'solution' and 'short answer' from LLM outputs.
    Includes fallback regex matching if standard JSON parsing fails.
    """
    if pd.isna(text):
        return None, None
        
    text = str(text).strip()
    
    # Strip markdown code block formatting if present
    if text.startswith('```json'):
        text = text[7:]
    elif text.startswith('```'):
        text = text[3:]
    if text.endswith('```'):
        text = text[:-3]
        
    text = text.strip()

    try:
        data = json.loads(text)
        solution = data.get('solution', '')
        short_ans = data.get('short answer', data.get('short_answer', ''))
        return str(short_ans), str(solution)
    except Exception:
        # Fallback: robust regex extraction if JSON is malformed or cut off
        short_ans, solution = "", ""
        
        # --- SHORT ANSWER ---
        # First try: captures everything after the opening quote until a closing quote or end-of-string
        sa_matches = re.findall(r'"?short[\s_\-]?answer"?\s*:\s*"([^"]*)', text, re.IGNORECASE)
        if sa_matches: 
            short_ans = sa_matches[-1].strip()
        else:
            # Second try: unquoted numbers/text
            sa_matches_unquoted = re.findall(r'"?short[\s_\-]?answer"?\s*:\s*([^"}\n,]+)', text, re.IGNORECASE)
            if sa_matches_unquoted:
                short_ans = sa_matches_unquoted[-1].strip()
                
        # --- SOLUTION ---
        # First try: captures everything after the opening quote until a closing quote or end-of-string
        sol_matches = re.findall(r'"?solution"?\s*:\s*"([^"]*)', text, re.IGNORECASE)
        if sol_matches: 
            solution = sol_matches[-1].strip()
        else:
            # Second try: unquoted fallback
            sol_matches_unquoted = re.findall(r'"?solution"?\s*:\s*([^"}\n,]+)', text, re.IGNORECASE)
            if sol_matches_unquoted:
                solution = sol_matches_unquoted[-1].strip()
                
        # Drop the trailing dot if the text was abruptly cut off
        if short_ans.endswith('.'):
            short_ans = short_ans[:-1]
        if solution.endswith('.'):
            solution = solution[:-1]
            
        return short_ans, solution

# ==========================================
# 2. GRADING FUNCTIONS
# ==========================================

def normalize_scientific(text):
    """Converts human scientific notation (e.g., 1.8×10^-7) to computer notation (1.8e-7)"""
    if pd.isna(text): return ""
    text = str(text).replace('×', 'x').replace('−', '-')
    # Matches 'x10^', '*10^', ' x 10^{-7}', etc., and converts to 'e'
    return re.sub(r'(?i)\s*[x\*]\s*10\s*\^[{]?([-+]?\d+)[}]?', r'e\1', text)

def extract_number(text):
    """
    Smarter extraction: Handles fractions, percentages, thousands separators, 
    unicode minus signs, and standard/scientific notation.
    """
    if pd.isna(text): return None
    text = str(text).strip()

    # Normalize scientific notation first!
    text = normalize_scientific(text)
    
    # 1. Fix typographical minus signs (convert to standard hyphen)
    text = text.replace('−', '-').replace('–', '-')
    
    # 2. Strip commas used as thousands separators (e.g., 1,000,000 -> 1000000)
    # Uses lookbehind/lookahead to only remove commas between digits
    # text = re.sub(r'(?<=\d),(?=\d{3})', '', text)
    
    # 3. Check for fractions (e.g., "1/2", "-5/3")
    fraction_match = re.search(r'([-+]?\d*\.\d+|[-+]?\d+)\s*/\s*([-+]?\d*\.\d+|[-+]?\d+)', text)
    if fraction_match:
        try:
            num = float(fraction_match.group(1))
            den = float(fraction_match.group(2))
            if den != 0: return num / den
        except:
            pass

    # 4. Check for percentages (e.g., "50%" -> 50.0)
    pct_match = re.search(r'([-+]?(?:\d*\.\d+|\d+)(?:[eE][-+]?\d+)?)\s*%', text)
    if pct_match:
        try:
            return float(pct_match.group(1))
        except:
            pass

    # 5. Standard float and scientific notation fallback
    match = re.search(r"[-+]?(?:\d*\.\d+|\d+)(?:[eE][-+]?\d+)?", text)
    if match:
        return float(match.group(0))
        
    return None

def compare_with_tolerance(pred_val, truth_str, tolerance_percent=1.0):
    """
    1. Finds how many decimal places the ground truth has.
    2. Rounds the prediction to match (skips if scientific notation).
    3. Uses percentage tolerance to check if it's correct.
    """
    truth_val = float(truth_str)
    
    # Handle the edge case where the ground truth is exactly 0
    if truth_val == 0:
        return pred_val == 0
        
    # NEW: If the ground truth uses scientific notation, skip rounding!
    if 'e' in truth_str.lower():
        error = abs(pred_val - truth_val) / abs(truth_val)
        return error <= (tolerance_percent / 100.0)
    
    # Dynamically find decimals in standard numbers (e.g., "0.452" -> 3)
    decimals = len(truth_str.split('.')[1]) if '.' in truth_str else 0
    
    # Round the prediction to match
    rounded_pred = round(pred_val, decimals)
    
    # Calculate percentage error and check against tolerance
    error = abs(rounded_pred - truth_val) / abs(truth_val)
    return error <= (tolerance_percent / 100.0)

def grade_short_answer(prediction, truth, tolerance_percent=1.0):
    """
    Grades short answer by extracting numbers and using compare_with_tolerance.
    Uses SymPy as a final fallback for symbolic/algebraic equivalence.
    """
    pred_str = str(prediction).strip()
    truth_str = str(truth).strip()
    
    # --- PHASE 1: Fast Float Extraction & Tolerance ---
    try:
        pred_val = extract_number(pred_str)
        
        if pred_val is None:
            # If no numbers found, check exact string first to save compute
            if pred_str.lower() == truth_str.lower(): 
                return True
        else:
            # Rounding and percentage tolerance
            if compare_with_tolerance(pred_val, truth_str, tolerance_percent):
                return True
    except Exception:
        pass  # If math conversion crashes, fall through to Phase 2

    # --- PHASE 2: SymPy Symbolic Fallback ---
    # This catches "1/3" == "2/6", "x+1" == "1+x", or "sqrt(2)"
    try:
        # Convert strings to SymPy math expressions
        expr_pred = sympy.sympify(pred_str)
        expr_truth = sympy.sympify(truth_str)
        
        # If subtracting them equals exactly 0, they are mathematically identical
        if sympy.simplify(expr_pred - expr_truth) == 0:
            return True
    except Exception:
        pass  # If SymPy can't parse the messy string, fall through to Phase 3

    # --- PHASE 3: Absolute Final String Fallback ---
    return pred_str.lower() == truth_str.lower()

def extract_mcq_option(text):
    """
    Extracts the multiple choice letter (A-E) from strings like 'A.', 'a)', 'Option A', 
    or 'A. Mitochondria'.
    """
    if pd.isna(text): 
        return ""
    text = str(text).strip().upper()
    
    # 1. Match "A.", "A)", "A:", or just "A " at the very beginning of the string
    match_start = re.search(r'^([A-E])(?:[\.\)\:\s]|$)', text)
    if match_start:
        return match_start.group(1)

    # 2. Fallback: return the original string in lowercase if no clean letter is found
    return text.strip().lower()

def grade_mcq(prediction, truth):
    pred_opt = extract_mcq_option(prediction)
    truth_opt = extract_mcq_option(truth)
    
    return pred_opt == truth_opt

def parse_duration_mins(text):
    """Helper to extract total minutes from strings like '3 hours 30 mins' or '03:30'"""
    h, m = 0.0, 0.0
    h_match = re.search(r'(\d+(?:\.\d+)?)\s*(?:hour|hours|hr|hrs|h)\b', text, re.IGNORECASE)
    m_match = re.search(r'(\d+(?:\.\d+)?)\s*(?:minute|minutes|min|mins|m)\b', text, re.IGNORECASE)
    
    if h_match: h = float(h_match.group(1))
    if m_match: m = float(m_match.group(1))
    
    # Fallback to digital 03:30 format
    if h == 0 and m == 0:
        dig = re.search(r'\b(\d{1,2}):(\d{2})\b', text)
        if dig:
            h, m = float(dig.group(1)), float(dig.group(2))
            
    return (h * 60.0) + m

def parse_line_equation(text):
    """Helper to extract slope (m) and intercept (c) from 'y = mx + c'"""
    clean_str = text.lower().replace(" ", "").replace("y=", "")
    m_match = re.search(r'([-+]?\d*(?:\.\d+)?(?:/\d+)?)\*?x', clean_str)
    c_match = re.search(r'(?<![x/])([-+]?\d+(?:\.\d+)?)(?![x/])', clean_str)
    
    m_val, c_val = None, None
    if m_match:
        m_str = m_match.group(1)
        if m_str in ['-', '']: m_val = -1.0 if m_str == '-' else 1.0
        elif '/' in m_str: 
            parts = m_str.split('/')
            m_val = float(parts[0]) / float(parts[1])
        else: m_val = float(m_str)
        
    if c_match:
        c_val = float(c_match.group(1))
        
    return m_val, c_val

def grade_free_form(prediction, truth, problem_id):
    pred_str = str(prediction).lower().strip()
    truth_str = str(truth).lower().strip()
    
    try:
        if problem_id == "Mat008_multi":
            pred_mins = parse_duration_mins(pred_str)
            truth_mins = parse_duration_mins(truth_str)
            
            if truth_mins > 0:
                return pred_mins == truth_mins # exact match check
                
        elif problem_id == "Mat011_multi":
            # Match 7:10, 07:10, 7.10
            pred_time = re.search(r'\b(0?[0-9]|1[0-9]|2[0-3])\s*[:\.]\s*([0-5][0-9])\b', pred_str)
            truth_time = re.search(r'\b(0?[0-9]|1[0-9]|2[0-3])\s*[:\.]\s*([0-5][0-9])\b', truth_str)
            
            if truth_time and pred_time:
                # Convert both to total minutes since midnight
                p_mins = int(pred_time.group(1)) * 60 + int(pred_time.group(2))
                t_mins = int(truth_time.group(1)) * 60 + int(truth_time.group(2))
                return p_mins == t_mins

        elif problem_id in ["Mat012_multi", "Mat013_multi", "Mat081_multi"]:
            # Vacuum all numbers from both strings
            pred_nums = [float(n) for n in re.findall(r'[-+]?\d*\.\d+|[-+]?\d+', pred_str)]
            truth_nums = [float(n) for n in re.findall(r'[-+]?\d*\.\d+|[-+]?\d+', truth_str)]
            
            if not truth_nums: return False
            
            # Check if every ground truth number is present in the LLM's prediction
            # (Allows the LLM to output things in different orders, like "x=4 or x=0")
            matched = 0
            for t in truth_nums:
                if any(abs(t - p) < 0.01 for p in pred_nums): # allow some errors
                    matched += 1
            return matched == len(truth_nums)

        elif problem_id == "Mat032_multi":
            pred_m, pred_c = parse_line_equation(pred_str)
            truth_m, truth_c = parse_line_equation(truth_str)
            
            if None not in (truth_m, truth_c, pred_m, pred_c):
                return abs(pred_m - truth_m) < 0.01 and abs(pred_c - truth_c) < 0.01 # allow some errors
                
        elif problem_id == "Mat080_multi":
            pred_nums = [float(n) for n in re.findall(r'[-+]?\d*\.\d+|[-+]?\d+', pred_str)]
            truth_nums = [float(n) for n in re.findall(r'[-+]?\d*\.\d+|[-+]?\d+', truth_str)]
            
            if len(truth_nums) == 4 and len(pred_nums) >= 4:
                # Compare the truth against the LAST 4 numbers in the prediction
                pred_last_4 = pred_nums[-4:]
                return all(abs(p - t) < 0.1 for p, t in zip(pred_last_4, truth_nums))
            
        elif problem_id in ["Phy060_multi", "Phy069_multi", "Phy070_multi", "Phy100_multi", "Phy117_multi", "Phy129_multi"]:
            
            # Clean common physics unit exponents so they don't break the number vacuum
            clean_p = pred_str.replace('^-1', '').replace('^{-1}', '').replace('s-1', '').replace('s -1', '')
            clean_t = truth_str.replace('^-1', '').replace('^{-1}', '').replace('s-1', '').replace('s -1', '')
            
            p_nums = [float(n) for n in re.findall(r'[-+]?\d*\.\d+|[-+]?\d+', clean_p)]
            # Keep truth as STRINGS so we can count the decimal places!
            t_strs = re.findall(r'[-+]?\d*\.\d+|[-+]?\d+', clean_t)
            
            if not t_strs or not p_nums:
                return False

            # Group 1: Extract 2 numbers and compare (Value & Uncertainty, Speed & Angle, etc.)
            if problem_id in ["Phy060_multi", "Phy070_multi", "Phy100_multi"]:
                if len(t_strs) >= 2 and len(p_nums) >= 2:
                    return all(compare_with_tolerance(p, t) for p, t in zip(p_nums[-2:], t_strs[-2:]))
                    
            # Group 2: Extract 1 number and a Left/Right direction
            elif problem_id == "Phy069_multi":
                t_dir = "left" if "left" in truth_str else ("right" if "right" in truth_str else "")
                p_dir = "left" if "left" in pred_str else ("right" if "right" in pred_str else "")
                return compare_with_tolerance(p_nums[-1], t_strs[-1]) and t_dir == p_dir
                
            # Group 3: Extract 1 number and an Up/Down direction
            elif problem_id == "Phy117_multi":
                t_dir = "up" if "up" in truth_str else ("down" if "down" in truth_str else "")
                p_dir = "up" if "up" in pred_str else ("down" if "down" in pred_str else "")
                return compare_with_tolerance(p_nums[-1], t_strs[-1]) and t_dir == p_dir
                
            # Group 4: Extract a Ratio (Simplified exact match for A:B format)
            elif problem_id == "Phy129_multi":
                t_nums = [float(n) for n in t_strs]
                
                # Check if we successfully extracted at least two numbers from both
                if len(t_nums) >= 2 and len(p_nums) >= 2:
                    # Compare the last two numbers directly (e.g., [1.0, 2.0] == [1.0, 2.0])
                    return p_nums[-2:] == t_nums[-2:]
                    
                return False
                
        elif problem_id == "Mat005_multi":
            def parse_bottle_choice(text):
                # 1. Erase any decimal/comma numbers (like 1.42 or 2.53) 
                # so they aren't accidentally read as '1' or '2'
                clean_text = re.sub(r'\d+[\.,]\d+', '', str(text).lower())
                
                # 2. Now safely match isolated 1, 2, a, or b
                matches = re.findall(r'\b(1|2|a|b)\b', clean_text)
                if matches:
                    last_val = matches[-1]
                    return 'A' if last_val in ['1', 'a'] else 'B'
                    
                # 3. Fallback: if they just output "2.53", it returns "2.53" which safely fails the check
                return str(text).strip().lower()

            return parse_bottle_choice(pred_str) == parse_bottle_choice(truth_str)

        # Computer Science - Exact Match for Binary/Hex
        elif problem_id in ["Com001_multi", "Com002_multi", "Com005_multi"]:
            # 1. Clean Pandas .0 artifacts, spaces, and prefixes
            t_clean = truth_str.lower().replace(".0", "").replace(" ", "")
            p_clean = pred_str.lower().replace(".0", "").replace("0b", "").replace("0x", "")
            
            # 2. Safely check if the exact number exists as a standalone word
            return bool(re.search(r'\b' + re.escape(t_clean) + r'\b', p_clean))

        # Computer Science - Logic Circuits (Booleans)
        elif problem_id in ["Com017_multi", "Com018_multi", "Com019_multi", "Com020_multi", 
                            "Com021_multi", "Com024_multi", "Com025_multi", "Com027_multi", "Com028_multi"]:
            
            # 1. Clean the dangerous ".0" float artifacts from both strings!
            t_str = truth_str[:-2] if truth_str.endswith('.0') else truth_str
            # We use replace on prediction in case the LLM wrote "The answer is 1.0"
            p_str = pred_str.replace(".0", "")
            
            # Define equivalence groups for digital logic
            true_vals = ['1', 'true', 'high', 'on']
            false_vals = ['0', 'false', 'low', 'off']
            
            # Determine the ground truth logical state
            t_is_true = t_str in true_vals
            
            # Find ALL logical keywords in the LLM's cleaned prediction
            matches = re.findall(r'\b(1|0|true|false|high|low|on|off)\b', p_str)
            
            if matches:
                # LLMs almost always put their final conclusion at the very end
                last_val = matches[-1]
                p_is_true = last_val in true_vals
                return p_is_true == t_is_true
                
            return False

    except Exception:
        pass  # If any mathematical parsing fails, gracefully fall through

    # Fallback for unhandled problem_ids or exact strings
    return pred_str == truth_str

def evaluate_answer(prediction, truth, question_type, problem_id):
    if pd.isna(prediction) or pd.isna(truth):
        return 0
        
    q_type = str(question_type).lower()
    
    if "mcq" in q_type:
        is_correct = grade_mcq(prediction, truth)
    elif "short" in q_type:
        is_correct = grade_short_answer(prediction, truth)
    else:
        is_correct = grade_free_form(prediction, truth, problem_id)
        
    return 1 if is_correct else 0

def highlight_correctness(val):
    """Colors the cell green or red based on hidden zero-width characters."""
    val_str = str(val)
    # \u200B is a zero-width space (invisible flag for Correct)
    if val_str.endswith('\u200B'):
        return 'background-color: #C6EFCE; color: #006100' # Green
    # \u200C is a zero-width non-joiner (invisible flag for Incorrect)
    elif val_str.endswith('\u200C'):
        return 'background-color: #FFC7CE; color: #9C0006' # Red
    return ''

In [ ]:
# # read master df
# PATH_TO_MASTER_DF = os.path.join('sample', 'master_dataset.xlsx')
# df_master = pd.read_excel(PATH_TO_MASTER_DF)
# df_master.head()

# # filter find only `Free-form` problem type
# df_master = df_master[df_master['question_type'] == 'Free-form']

# # extract df_master free form only to xlsx
# PATH_TO_FREEFORM_DF = os.path.join('sample', 'master_dataset_freeform.xlsx')
# df_master.to_excel(PATH_TO_FREEFORM_DF, index=False)
# print(f"[INFO] Saved filtered df into {PATH_TO_FREEFORM_DF}")

# df_master.head()

In [ ]:
# ==========================================
# 3. MAIN WORKFLOW
# ==========================================
def main():
    print("[INFO] Starting evaluation pipeline...")
    
    out_dir = Path("result/process")
    out_dir.mkdir(parents=True, exist_ok=True)
    
    # Define mapping of response columns to languages
    lang_mapping = {
        'response': 'English',
        'response_arabic': 'Arabic',
        'response_chinese': 'Chinese',
        'response_hindi': 'Hindi',
        'response_indonesian': 'Indonesian',
        'response_kazakh': 'Kazakh',
        'response_swahili': 'Swahili'
    }
    
    # 1. LOAD EVERY FILE
    base_dir = Path("result/raw")
    all_dataframes = []
    
    if not base_dir.exists():
        print(f"[!] Directory {base_dir} not found. Please check your path.")
        return

    xlsx_files = list(base_dir.rglob("*.xlsx"))
    for xlsx_file in tqdm(xlsx_files, desc="Loading Excel files"):
        parts = xlsx_file.parts
        if len(parts) >= 4:
            subject = parts[-3]
            instance = parts[-2]
            model_name = xlsx_file.stem # Infer model name directly from file name
            
            try:
                df = pd.read_excel(xlsx_file, engine='openpyxl')
                df['subject'] = subject
                df['instance'] = instance
                df['model'] = model_name
                all_dataframes.append(df)
            except Exception as e:
                print(f"[!] Error reading {xlsx_file}: {e}")
                
    if not all_dataframes:
        print("[!] No data loaded. Exiting.")
        return
        
    master_df = pd.concat(all_dataframes, ignore_index=True)
    print(f"[INFO] Loaded {len(master_df)} total rows.")

    # ==========================================
    # 1.5 EXCLUDE SPECIFIC PROBLEMS
    # ==========================================
    excluded_file = Path("excluded_class.txt")
    excluded_problems = set()
    if excluded_file.exists():
        with open(excluded_file, 'r', encoding='utf-8') as f:
            excluded_problems = set(line.strip() for line in f if line.strip())
        print(f"[INFO] Loaded {len(excluded_problems)} problem IDs to exclude.")
        
        # Filter out any rows where the problem_id is in the exclusion set
        original_len = len(master_df)
        master_df = master_df[~master_df['problem_id'].isin(excluded_problems)]
        print(f"[INFO] Dropped {original_len - len(master_df)} excluded rows. Remaining: {len(master_df)} rows.")
    else:
        print(f"[INFO] No excluded_class.txt found. Skipping exclusion step.")

    # 2. EXTRACT JSON & 3. GRADE EACH PROBLEM
    print("[INFO] Extracting JSON and grading responses...")

    # 2. EXTRACT JSON & 3. GRADE EACH PROBLEM
    print("[INFO] Extracting JSON and grading responses...")
    eval_records = []
    
    for index, row in tqdm(master_df.iterrows(), total=len(master_df), desc="Extracting & Grading"):
        truth = row.get('answer', '')
        q_type = row.get('question_type', 'Short Answer')
        model = row['model']
        subject = row['subject'].capitalize()
        problem_id = row.get('problem_id', 'Unknown')
        instance = row.get('instance', 'Unknown')
        
        for col_name, lang_name in lang_mapping.items():
            if col_name in row:
                raw_response = row[col_name]
                short_ans, solution = extract_json_fields(raw_response)
                score = evaluate_answer(short_ans, truth, q_type, problem_id)
                
                # Truncate to 150 chars just in case the LLM hallucinated a massive paragraph
                ans_display = str(short_ans).strip()[:150] if short_ans else "NO ANSWER"
                # Attach invisible markers to the end of the text
                display_val = f"{ans_display}\u200B" if score == 1 else f"{ans_display}\u200C"
                
                eval_records.append({
                    'Model': model,
                    'Subject': subject,
                    'Language': lang_name,
                    'Problem_ID': problem_id,
                    'Instance': instance,
                    'Accuracy': score,
                    'Display_Value': display_val  # <--- Added this!
                })

    eval_df = pd.DataFrame(eval_records)

    # 4. CALCULATE METRICS
    print("[INFO] Computing Metrics...")

    # Metric 1: Average Accuracy
    avg_df = eval_df.groupby(['Model', 'Subject', 'Language'])['Accuracy'].mean().reset_index()
    avg_pivot = avg_df.pivot_table(index='Model', columns=['Subject', 'Language'], values='Accuracy')

    # Metric 2: Worst-Case Accuracy
    # Group by Problem_ID first to find if *any* instance failed (min = 0 if any failed)
    wc_prob_df = eval_df.groupby(['Model', 'Subject', 'Language', 'Problem_ID'])['Accuracy'].min().reset_index()
    wc_df = wc_prob_df.groupby(['Model', 'Subject', 'Language'])['Accuracy'].mean().reset_index()
    wc_pivot = wc_df.pivot_table(index='Model', columns=['Subject', 'Language'], values='Accuracy')

    # Metric 3: Reasoning Robustness
    # safely handle 0/0 (NaN) and 1/0 (inf)
    robustness_pivot = (wc_pivot / avg_pivot).replace([np.inf, -np.inf], np.nan).fillna(0)

    # ==========================================
    # 4.5 PARTITION ANALYSIS (VARIABLES VS NO VARIABLES)
    # ==========================================
    print("[INFO] Performing Partition Analysis (Variables vs No Variables)...")

    # Load problem IDs that contain variables
    var_class_file = Path("id_with_var.txt")
    var_problems = set()
    if var_class_file.exists():
        with open(var_class_file, 'r', encoding='utf-8') as f:
            var_problems = set(line.strip() for line in f if line.strip())
    else:
        print(f"[WARNING] {var_class_file} not found. Partitioning will be skipped/empty.")

    # Tag each record in the master dataframe
    eval_df['Has_Variables'] = eval_df['Problem_ID'].apply(lambda x: 'Yes' if x in var_problems else 'No')

    # Step A: Get the Average and Worst-Case Accuracy for each Language within the partitions
    part_avg_lang = eval_df.groupby(['Model', 'Subject', 'Language', 'Has_Variables'])['Accuracy'].mean().reset_index()
    
    part_wc_prob = eval_df.groupby(['Model', 'Subject', 'Language', 'Has_Variables', 'Problem_ID'])['Accuracy'].min().reset_index()
    part_wc_lang = part_wc_prob.groupby(['Model', 'Subject', 'Language', 'Has_Variables'])['Accuracy'].mean().reset_index()

    # Step B: Calculate Mean and Std Dev ACROSS languages for those partitions
    part_avg_stats = part_avg_lang.groupby(['Model', 'Subject', 'Has_Variables'])['Accuracy'].agg(['mean', 'std']).reset_index()
    part_avg_stats.rename(columns={'mean': 'Avg_Acc_Mean', 'std': 'Avg_Acc_Std'}, inplace=True)

    part_wc_stats = part_wc_lang.groupby(['Model', 'Subject', 'Has_Variables'])['Accuracy'].agg(['mean', 'std']).reset_index()
    part_wc_stats.rename(columns={'mean': 'WC_Acc_Mean', 'std': 'WC_Acc_Std'}, inplace=True)

    # Step C: Merge and Pivot into a highly readable side-by-side format
    partition_metrics = pd.merge(part_avg_stats, part_wc_stats, on=['Model', 'Subject', 'Has_Variables'])
    
    # Pivot so "Yes" and "No" become side-by-side columns
    partition_pivot = partition_metrics.pivot(index=['Model', 'Subject'], columns='Has_Variables')
    
    # Flatten the multi-level column names for Excel (e.g., 'Avg_Acc_Mean_Yes', 'WC_Acc_Std_No')
    partition_pivot.columns = [f"{metric}_{var_status}" for metric, var_status in partition_pivot.columns]
    partition_pivot.reset_index(inplace=True)

    # 5. EXPORT ARTIFACTS
    print(f"[INFO] Saving artifacts to {out_dir}...")
    
    avg_acc_path = out_dir / "average_accuracy.xlsx"
    wc_acc_path = out_dir / "worst_case_accuracy.xlsx"
    
    # Apply .round(2) right as you export to Excel
    avg_pivot.round(2).to_excel(avg_acc_path)
    wc_pivot.round(2).to_excel(wc_acc_path)
    
    # --- NEW EXPORT ---
    partition_path = out_dir / "variable_partition_metrics.xlsx"
    partition_pivot.round(4).to_excel(partition_path, index=False)
    
    print(f"  -> Saved {avg_acc_path}")
    print(f"  -> Saved {wc_acc_path}")
    print(f"  -> Saved {partition_path}") # <--- Print the new save
    
    print(f"  -> Saved {avg_acc_path}")
    print(f"  -> Saved {wc_acc_path}")

    # ==========================================
    # 6. EXPORT DETAILED 'ANALYZED' FOLDERS
    # ==========================================
    print("[INFO] Generating detailed analyzed spreadsheets...")
    analyzed_dir = out_dir / "analyzed"
    
    # Pivot the data: Rows = Problem/Language, Columns = Instances
    detail_pivot = eval_df.pivot_table(
        index=['Model', 'Subject', 'Language', 'Problem_ID'],
        columns='Instance',
        values='Display_Value',   # <--- Change this from 'Accuracy'
        aggfunc='first'           # <--- Add this so Pandas knows how to pivot strings!
    ).reset_index()

    # Identify which columns are the "instance" columns dynamically
    instance_cols = [col for col in detail_pivot.columns if col not in ['Model', 'Subject', 'Language', 'Problem_ID']]

    # Convert groupby to a list so tqdm knows the total number of models
    grouped_models = list(detail_pivot.groupby('Model'))
    for model_name, model_df in tqdm(grouped_models, desc="Exporting analyzed sheets"):
        model_dir = analyzed_dir / model_name
        model_dir.mkdir(parents=True, exist_ok=True)
        
        for subject, subj_df in model_df.groupby('Subject'):
            file_path = model_dir / f"{subject.lower()}.xlsx"
            
            # Use ExcelWriter to allow multiple sheets in the same file
            with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
                
                # Group by Language to create a new sheet for each
                for language, lang_df in subj_df.groupby('Language'):
                    
                    # Drop columns that are no longer needed for display
                    out_df = lang_df.drop(columns=['Model', 'Subject', 'Language'])
                    
                    # Apply the red/green colors just to the instance columns
                    # Note: If you use Pandas older than 2.1.0, change .map to .applymap
                    styled_df = out_df.style.map(highlight_correctness, subset=instance_cols)
                    
                    # Save the dataframe to its specific language sheet
                    styled_df.to_excel(writer, sheet_name=language, index=False)
            
    print(f"  -> Saved detailed spreadsheets to {analyzed_dir}")

    # ==========================================
    # 7. GENERATE REASONING ROBUSTNESS CHART
    # ==========================================
    print("[INFO] Generating Reasoning Robustness chart...")
    
    # Flatten the pivot table and average across languages for chart readability
    plot_df = robustness_pivot.stack(level=['Subject', 'Language']).reset_index(name='Robustness')
    plot_data = plot_df.groupby(['Model', 'Subject'])['Robustness'].mean().reset_index()

    # Set up a neat, wide figure
    plt.figure(figsize=(14, 7))
    sns.set_theme(style="whitegrid")
    
    # Create the grouped bar chart
    ax = sns.barplot(
        data=plot_data, 
        x='Subject', 
        y='Robustness', 
        hue='Model', 
        palette='viridis'
    )
    
    # Clean styling and labels
    plt.title('Reasoning Robustness by Subject and Model\n(Averaged across languages)', fontsize=16, pad=20, fontweight='bold')
    plt.ylabel('Reasoning Robustness', fontsize=12, fontweight='bold')
    plt.xlabel('Subject', fontsize=12, fontweight='bold')
    plt.ylim(0, 1.1)  # Expand Y-axis slightly so labels don't get cut off
    
    # Move the legend entirely outside the chart so it doesn't block the bars
    plt.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
    
    # Neatly annotate the exact values on top of every bar
    for p in ax.patches:
        height = p.get_height()
        if pd.notnull(height) and height > 0: # Only label bars that actually have a value
            ax.annotate(f'{height:.2f}', 
                       (p.get_x() + p.get_width() / 2., height), 
                       ha='center', va='bottom', 
                       xytext=(0, 4), 
                       textcoords='offset points',
                       fontsize=10)
                   
    # Save the plot
    plt.tight_layout()
    chart_path = out_dir / "reasoning_robustness.png"
    plt.savefig(chart_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  -> Saved chart to {chart_path}")

    # ==========================================
    # 8. EXPORT HARD PROBLEMS ANALYSIS TEMPLATE
    # ==========================================
    print("[INFO] Generating hard problems analysis template...")
    target_model = "gpt-5_4-2026-03-05"
    failure_threshold = 0

    # 1. Isolate the data for the target model AND filter for English only
    gpt_df = eval_df[(eval_df['Model'] == target_model) & (eval_df['Language'] == 'English')]

    if not gpt_df.empty:
        # 2. Calculate the failure rate for each Problem ID within each Subject
        # Since Accuracy is 1 or 0, failure rate is simply 1.0 - Mean Accuracy
        prob_stats = gpt_df.groupby(['Subject', 'Problem_ID'])['Accuracy'].mean().reset_index()
        prob_stats['Failure_Rate'] = 1.0 - prob_stats['Accuracy']
        
        # 3. Filter for problems where the failure rate is greater than the threshold (> 80%)
        hard_probs = prob_stats[prob_stats['Failure_Rate'] > failure_threshold]
        
        analysis_path = out_dir / "hard_problems_analysis.xlsx"
        
        # 4. Write to Excel, splitting by Subject into different sheets
        with pd.ExcelWriter(analysis_path, engine='openpyxl') as writer:
            for subject, subj_df in hard_probs.groupby('Subject'):
                
                # --- NEW: Sort by Accuracy mathematically (Lowest pass rate at the top) ---
                subj_df = subj_df.sort_values(by='Accuracy', ascending=True)
                
                # Create the requested layout with Pass Rate included
                out_sheet_df = pd.DataFrame({
                    'Problem ID': subj_df['Problem_ID'],
                    'Pass Rate': (subj_df['Accuracy'] * 100).round(1).astype(str) + '%', # Converts 0.15 to "15.0%"
                    'Verdict': '',
                    'Analysis': ''
                })
                
                # Save to a sheet named after the Subject
                out_sheet_df.to_excel(writer, sheet_name=subject, index=False)
                
        print(f"  -> Saved hard problems template to {analysis_path}")
    else:
        print(f"[WARNING] Target model '{target_model}' not found in the dataset. Skipping template generation.")

if __name__ == "__main__":
    main()